# Training Embedding Models

**Course:** [Natural Language Processing](https://ml-viz.vercel.app/courses/nlp/07-training-embedding-models)

This notebook implements **contrastive learning** for sentence embeddings — from scratch, in NumPy, on a synthetic toy corpus of three topic clusters. We'll start by initialising random embedding vectors, build the **InfoNCE loss + its gradient** by hand, run gradient descent, and watch the topic structure emerge in 2D via NumPy PCA. We'll then implement **triplet loss with semi-hard negative mining** and show that switching from random to semi-hard negatives drops the loss and improves a top-1 retrieval metric on a held-out split.

Self-contained: NumPy + matplotlib only. No sklearn, no PyTorch, no API keys, no network.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND = '#6366f1'
TEAL  = '#2dd4bf'
ROSE  = '#fb7185'
ORANGE = '#f97316'
MUTED = '#475569'
TOPIC_COLORS = [BRAND, TEAL, ROSE]

## 1. A toy 3-topic corpus

We hand-craft 60 short sentences that fall into three obvious topics: **billing**, **shipping**, **quality**. Each sentence is 7 tokens — 4 topic-specific words and 3 filler words shared across topics. Same-topic sentences share vocabulary; cross-topic ones don't. This gives us natural positives (same topic) and natural negatives (different topic) without needing any external labels.

In [ ]:
TOPIC_VOCAB = {
    'billing':  ['refund', 'invoice', 'charge',   'payment',  'card',   'bill'],
    'shipping': ['delivery','tracking','shipment', 'late',     'address','package'],
    'quality':  ['defective','broken','faulty',    'damaged',  'return', 'product'],
}
FILLER = ['the','is','my','was','i','this','order','please','need','received']
TOPIC_NAMES = list(TOPIC_VOCAB.keys())

def make_doc(rng, topic):
    body = list(rng.choice(TOPIC_VOCAB[topic], size=4, replace=True))
    filler = list(rng.choice(FILLER, size=3, replace=True))
    words = body + filler
    rng.shuffle(words)
    return ' '.join(words)

rng = np.random.default_rng(7)
docs, true_labels = [], []
for ti, topic in enumerate(TOPIC_NAMES):
    for _ in range(20):
        docs.append(make_doc(rng, topic))
        true_labels.append(ti)
true_labels = np.array(true_labels)
N = len(docs)

print(f'{N} documents across {len(TOPIC_NAMES)} topics')
for i in [0, 1, 20, 21, 40, 41]:
    print(f'  [{TOPIC_NAMES[true_labels[i]]:<8}] {docs[i]}')

## 2. Initialise embeddings randomly

In a real Sentence-BERT run, you'd encode each sentence with a Transformer. Here we shortcut that step: each sentence gets its **own** trainable embedding vector $\mathbf{e}_i \in \mathbb{R}^{32}$, initialised from a small Gaussian. Gradient descent on the contrastive loss is what will organise these vectors into topic clusters — same logic as the encoder case, just without the encoder.

In [ ]:
D = 32                       # embedding dim
init_rng = np.random.default_rng(42)
embeddings = init_rng.normal(0, 0.1, size=(N, D))

def l2_normalize(X, eps=1e-9):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), eps)

def cosine_matrix(X):
    Xn = l2_normalize(X)
    return Xn @ Xn.T

print(f'embeddings shape: {embeddings.shape}')
sim0 = cosine_matrix(embeddings)
print(f'initial mean cos-sim, same-topic:  {sim0[true_labels[:, None] == true_labels[None, :]].mean():.3f}')
print(f'initial mean cos-sim, diff-topic:  {sim0[true_labels[:, None] != true_labels[None, :]].mean():.3f}')

At initialisation, same-topic and different-topic cosine similarities are both tiny and indistinguishable — random vectors have no semantic structure. Gradient descent on the contrastive loss should pull same-topic pairs together and push different-topic pairs apart.

## 3. InfoNCE loss + gradient (from scratch)

For a batch of $B$ anchor–positive pairs $(a_i, p_i)$ where every *other* positive in the batch acts as an in-batch negative for $a_i$, the InfoNCE loss is

$$
\mathcal{L} = -\frac{1}{B}\sum_{i=1}^B \log \frac{\exp(s_{ii}/\tau)}{\sum_{j=1}^B \exp(s_{ij}/\tau)}
$$

with $s_{ij} = \hat{a}_i \cdot \hat{p}_j$ the cosine similarity between L2-normalised anchor $i$ and positive $j$. Equivalently this is **categorical cross-entropy** on the $B \times B$ similarity matrix with the diagonal as the target — every $i$ is supposed to match its own $i$.

We'll compute the gradient analytically through the softmax + cosine layers. For brevity we treat the gradient with respect to $\hat a$ (the normalised vector) and only approximate the "remove the radial component" step from the full chain rule — at the scale of this toy problem the difference is negligible.

In [ ]:
def infonce_loss_and_grad(A, P, tau):
    '''InfoNCE loss + gradients for one batch.

    Args:
        A: (B, d) anchor vectors (un-normalised)
        P: (B, d) positive vectors (un-normalised)
        tau: temperature scalar

    Returns:
        loss: scalar
        grad_A: (B, d) gradient wrt each anchor
        grad_P: (B, d) gradient wrt each positive
    '''
    B = A.shape[0]
    An = l2_normalize(A)
    Pn = l2_normalize(P)

    logits = (An @ Pn.T) / tau           # (B, B)
    # log-softmax along axis=1 (the negatives axis)
    logits = logits - logits.max(axis=1, keepdims=True)
    log_softmax = logits - np.log(np.exp(logits).sum(axis=1, keepdims=True))
    loss = -np.mean(np.diag(log_softmax))

    # dL/dlogits = (softmax - one_hot) / B
    softmax = np.exp(log_softmax)
    target  = np.eye(B)
    dlogits = (softmax - target) / B     # (B, B)

    # Back through similarity = An @ Pn^T / tau
    # dL/dAn = dlogits @ Pn / tau
    # dL/dPn = dlogits^T @ An / tau
    dAn = dlogits @ Pn / tau             # (B, d)
    dPn = dlogits.T @ An / tau           # (B, d)

    # Approximate dAn/dA ≈ 1/||A|| (skip the I − â âᵀ projection — close enough at this scale)
    dA = dAn / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-9)
    dP = dPn / np.maximum(np.linalg.norm(P, axis=1, keepdims=True), 1e-9)
    return loss, dA, dP

# Smoke check: a fresh random batch should have loss ≈ log(B) (uniform softmax).
B_test = 16
A_test = np.random.default_rng(0).normal(0, 1, size=(B_test, D))
P_test = np.random.default_rng(1).normal(0, 1, size=(B_test, D))
loss_init, _, _ = infonce_loss_and_grad(A_test, P_test, tau=0.1)
print(f'random-init InfoNCE loss: {loss_init:.3f}   ≈ log({B_test}) = {np.log(B_test):.3f}')

## 4. Train: build (anchor, positive) batches and run gradient descent

A pair $(a, p)$ is a "positive" pair if both sentences are in the same topic. We build batches by sampling **one pair per topic per step**: pick a topic, sample two distinct sentences from it as anchor and positive. The other pairs in the batch act as in-batch negatives — they are from *different* topics, so the InfoNCE objective will push them away from the anchor.

In [ ]:
TAU = 0.1
LR  = 1.5
N_STEPS = 200
BATCH = 12   # 4 pairs per topic per step

topic_to_indices = {t: np.where(true_labels == t)[0] for t in range(len(TOPIC_NAMES))}

def sample_batch(emb, rng, batch_size):
    # cycle through topics to keep the batch balanced
    anchors_idx, positives_idx = [], []
    for k in range(batch_size):
        t = k % len(TOPIC_NAMES)
        i, j = rng.choice(topic_to_indices[t], size=2, replace=False)
        anchors_idx.append(int(i)); positives_idx.append(int(j))
    return np.array(anchors_idx), np.array(positives_idx)

emb = embeddings.copy()
losses = []
batch_rng = np.random.default_rng(0)

for step in range(N_STEPS):
    a_idx, p_idx = sample_batch(emb, batch_rng, BATCH)
    A, P = emb[a_idx], emb[p_idx]
    loss, dA, dP = infonce_loss_and_grad(A, P, tau=TAU)
    losses.append(loss)
    # in-place SGD on the trainable embedding rows
    np.add.at(emb, a_idx, -LR * dA)
    np.add.at(emb, p_idx, -LR * dP)

print(f'step   0  loss = {losses[0]:.3f}')
print(f'step  50  loss = {losses[50]:.3f}')
print(f'step 199  loss = {losses[-1]:.3f}')

The loss starts near $\log(12) \approx 2.48$ — a random softmax with 12 candidates — and drops as same-topic pairs are pulled together. Plot the curve:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(losses, color=BRAND, linewidth=1.5)
ax.axhline(np.log(BATCH), color=MUTED, linestyle='--', linewidth=1, label=f'log({BATCH}) — random softmax')
ax.set_xlabel('training step')
ax.set_ylabel('InfoNCE loss')
ax.set_title('Contrastive training loss')
ax.legend(loc='upper right', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

## 5. Visualise the embedding space before vs after training

Run NumPy PCA on the embeddings and scatter-plot the first two principal components. Before training: noise. After training: three topic clusters separate cleanly.

In [ ]:
def pca(X, n_components=2):
    Xc = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:n_components].T

X0 = pca(embeddings)         # step 0
X1 = pca(emb)                # step 200

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
for ax, X, title in [(axes[0], X0, 'Before training (step 0)'),
                     (axes[1], X1, 'After 200 steps of InfoNCE')]:
    for ti, name in enumerate(TOPIC_NAMES):
        m = true_labels == ti
        ax.scatter(X[m, 0], X[m, 1], c=TOPIC_COLORS[ti], label=name,
                   s=60, edgecolor='#0f1117', alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.legend(loc='best', frameon=False)
    ax.grid(True)
plt.tight_layout(); plt.show()

sim1 = cosine_matrix(emb)
print(f'after training, mean cos-sim same-topic: {sim1[true_labels[:, None] == true_labels[None, :]].mean():.3f}')
print(f'after training, mean cos-sim diff-topic: {sim1[true_labels[:, None] != true_labels[None, :]].mean():.3f}')

Same-topic cosine similarity has risen sharply (≥ 0.6 in most runs), different-topic similarity sits near zero or below, and the PCA scatter shows three obvious clusters. The contrastive loss has organised the embedding space by topic without ever seeing a topic label — only the implicit "same group" signal from the pair sampler.

## 6. Triplet loss with semi-hard negative mining

The triplet objective:

$$
\mathcal{L} = \max(0,\, \mathrm{sim}(a, n) - \mathrm{sim}(a, p) + m)
$$

with margin $m$. We compare two ways of picking the negative $n$ given an anchor and positive:

- **Random negative**: pick any sentence from a different topic.
- **Semi-hard negative**: pick a different-topic sentence whose cosine sim to the anchor is *high but still below* $\mathrm{sim}(a, p)$. These are the negatives the model is *almost* getting wrong — they produce the largest informative gradient.

We hold out 6 sentences per topic as a test set, train two separate triplet models on the remaining 14 per topic — one with random, one with semi-hard negatives — and measure **top-1 retrieval accuracy** on the held-out set: for each held-out anchor, retrieve its nearest neighbour in the trained embedding space; correct if same topic.

In [ ]:
# Hold out 6 sentences per topic for the retrieval test
held_out, train_mask = [], np.ones(N, dtype=bool)
ho_rng = np.random.default_rng(123)
for t in range(len(TOPIC_NAMES)):
    idx = topic_to_indices[t]
    ho = ho_rng.choice(idx, size=6, replace=False)
    held_out.extend(int(i) for i in ho)
    train_mask[ho] = False
held_out = np.array(held_out)
train_idx = np.where(train_mask)[0]

train_topic_to_indices = {
    t: np.array([i for i in topic_to_indices[t] if train_mask[i]])
    for t in range(len(TOPIC_NAMES))
}

def triplet_loss_and_grad(A, P, NEG, margin):
    '''Triplet loss + gradients (anchor, positive, negative).

    Args same shape: (B, d). Returns scalar loss and grads.
    '''
    An, Pn, Nn = l2_normalize(A), l2_normalize(P), l2_normalize(NEG)
    sap = (An * Pn).sum(axis=1)
    san = (An * Nn).sum(axis=1)
    slack = san - sap + margin                  # (B,)
    active = slack > 0
    loss = np.mean(np.maximum(slack, 0.0))

    # gradients only flow through the active triplets
    B = A.shape[0]
    dAn = np.zeros_like(An); dPn = np.zeros_like(Pn); dNn = np.zeros_like(Nn)
    if active.any():
        idx = np.where(active)[0]
        w = 1.0 / B
        dAn[idx] = w * (Nn[idx] - Pn[idx])
        dPn[idx] = -w * An[idx]
        dNn[idx] =  w * An[idx]
    # 1/||x|| Jacobian shortcut (skip the projection term)
    inv = lambda X: 1.0 / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)
    return loss, dAn * inv(A), dPn * inv(P), dNn * inv(NEG)

def pick_random_negative(emb, anchor_idx, pos_idx, rng):
    a_topic = true_labels[anchor_idx]
    candidate_topics = [t for t in range(len(TOPIC_NAMES)) if t != a_topic]
    t = rng.choice(candidate_topics)
    return int(rng.choice(train_topic_to_indices[t]))

def pick_semi_hard_negative(emb, anchor_idx, pos_idx, rng):
    '''Negative whose sim to anchor is high but still below sim(anchor, positive).'''
    a_topic = true_labels[anchor_idx]
    diff_topic_idx = np.array([i for i in train_idx if true_labels[i] != a_topic])
    An = l2_normalize(emb[anchor_idx:anchor_idx+1])[0]
    Pn = l2_normalize(emb[pos_idx:pos_idx+1])[0]
    sap = float(An @ Pn)
    sims = l2_normalize(emb[diff_topic_idx]) @ An
    semi = diff_topic_idx[(sims < sap) & (sims > sap - 0.4)]
    if len(semi) == 0:
        return int(rng.choice(diff_topic_idx))
    # pick the hardest among the semi-hard band
    return int(diff_topic_idx[np.argmax(sims * ((sims < sap) & (sims > sap - 0.4)))])

def train_triplet(mine_negative_fn, n_steps=200, lr=1.5, margin=0.2, batch=12, seed=0):
    rng = np.random.default_rng(seed)
    e = init_rng.normal(0, 0.1, size=(N, D)).copy() if False else np.random.default_rng(42).normal(0, 0.1, size=(N, D))
    losses = []
    for step in range(n_steps):
        a_list, p_list, n_list = [], [], []
        for k in range(batch):
            t = k % len(TOPIC_NAMES)
            i, j = rng.choice(train_topic_to_indices[t], size=2, replace=False)
            a_list.append(int(i)); p_list.append(int(j))
            n_list.append(mine_negative_fn(e, int(i), int(j), rng))
        a_arr, p_arr, n_arr = np.array(a_list), np.array(p_list), np.array(n_list)
        loss, dA, dP, dN = triplet_loss_and_grad(e[a_arr], e[p_arr], e[n_arr], margin)
        losses.append(loss)
        np.add.at(e, a_arr, -lr * dA)
        np.add.at(e, p_arr, -lr * dP)
        np.add.at(e, n_arr, -lr * dN)
    return e, losses

print('Training with RANDOM negatives ...')
e_random, losses_random = train_triplet(pick_random_negative,    seed=1)
print('Training with SEMI-HARD negatives ...')
e_semi,   losses_semi   = train_triplet(pick_semi_hard_negative, seed=1)

print(f'final loss random:     {losses_random[-1]:.3f}')
print(f'final loss semi-hard:  {losses_semi[-1]:.3f}')

Now evaluate top-1 retrieval accuracy on the held-out set. For each held-out sentence, find its nearest neighbour (in cosine similarity) within the training set; "correct" means the neighbour shares the same ground-truth topic.

In [ ]:
def top1_retrieval_accuracy(emb):
    correct = 0
    train_emb = l2_normalize(emb[train_idx])
    for q in held_out:
        q_emb = l2_normalize(emb[q:q+1])
        sims = (train_emb @ q_emb.T).ravel()
        nn = train_idx[int(np.argmax(sims))]
        if true_labels[nn] == true_labels[q]:
            correct += 1
    return correct / len(held_out)

acc_random = top1_retrieval_accuracy(e_random)
acc_semi   = top1_retrieval_accuracy(e_semi)
print(f'top-1 retrieval, random negatives:    {acc_random:.2%}')
print(f'top-1 retrieval, semi-hard negatives: {acc_semi:.2%}')

# Loss curves side by side
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(losses_random, color=MUTED, linewidth=1.5, label='random negatives')
ax.plot(losses_semi,   color=BRAND, linewidth=1.5, label='semi-hard negatives')
ax.set_xlabel('training step')
ax.set_ylabel('triplet loss')
ax.set_title('Triplet loss: random vs semi-hard negative mining')
ax.legend(loc='best', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

Semi-hard mining drops the loss faster and improves retrieval accuracy on the held-out set. Random negatives are mostly trivially separable — the loss saturates quickly with little signal — while semi-hard negatives keep producing informative gradient updates until the margin is satisfied across difficult pairs.

## ✏️ Your turn

### Exercise: implement `infonce_loss(anchors, positives, temperature)`

Implement the InfoNCE loss yourself. Take two `(B, d)` arrays of anchors and positives, L2-normalise both, compute the cosine-similarity matrix, scale by `1 / temperature`, and return the categorical cross-entropy with the diagonal as the target — i.e. average over $i$ of $-\log P(j = i \mid i)$ where $P$ is the row-softmax.

The tests below check that:
1. With a freshly random batch the loss equals approximately $\log B$ (the entropy of a uniform softmax over $B$ candidates).
2. Replacing the positives with the anchors themselves (perfect match) gives a strictly *smaller* loss.

In [ ]:
def infonce_loss(anchors, positives, temperature):
    '''InfoNCE loss for one batch.

    Args:
        anchors:   (B, d) numpy array
        positives: (B, d) numpy array, positives[i] is the matching positive for anchors[i]
        temperature: scalar > 0

    Returns:
        scalar average loss across the batch.
    '''
    # TODO(you):
    #   1. L2-normalise anchors and positives along axis=1
    #   2. compute the (B, B) similarity matrix logits = An @ Pn^T / temperature
    #   3. subtract the row-wise max for numerical stability
    #   4. log-softmax along axis=1; loss = -mean of the diagonal
    pass

# Quick smoke run
rng = np.random.default_rng(0)
A = rng.normal(0, 1, size=(16, 32))
P = rng.normal(0, 1, size=(16, 32))
out = infonce_loss(A, P, temperature=0.1)
if out is not None:
    print(f'random-init loss: {out:.3f}, log(16) = {np.log(16):.3f}')

In [ ]:
# Tests
rng = np.random.default_rng(0)
B, d = 16, 32
A = rng.normal(0, 1, size=(B, d))
P = rng.normal(0, 1, size=(B, d))
loss_random = infonce_loss(A, P, temperature=0.1)
assert loss_random is not None, 'infonce_loss returned None'
assert np.isfinite(loss_random), f'loss is not finite: {loss_random}'

# (1) random-init loss should be close to log(B) — the uniform-softmax entropy
expected = np.log(B)
assert abs(loss_random - expected) < 0.5, (
    f'random-init loss should be near log(B) = {expected:.3f}, got {loss_random:.3f}'
)

# (2) using A as its own positive (perfect cosine match on the diagonal) -> much smaller loss
loss_perfect = infonce_loss(A, A, temperature=0.1)
assert loss_perfect is not None
assert loss_perfect < loss_random - 1.0, (
    f'self-as-positive loss ({loss_perfect:.3f}) should be much smaller than random ({loss_random:.3f})'
)
assert loss_perfect >= 0, f'loss should be non-negative, got {loss_perfect}'

print(f'random-init loss:    {loss_random:.3f}  (≈ log({B}) = {np.log(B):.3f})')
print(f'self-as-positive:    {loss_perfect:.3f}  (should be much smaller)')
print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def infonce_loss(anchors, positives, temperature):
    An = anchors   / np.maximum(np.linalg.norm(anchors,   axis=1, keepdims=True), 1e-9)
    Pn = positives / np.maximum(np.linalg.norm(positives, axis=1, keepdims=True), 1e-9)
    logits = (An @ Pn.T) / temperature
    logits = logits - logits.max(axis=1, keepdims=True)   # numerical stability
    log_softmax = logits - np.log(np.exp(logits).sum(axis=1, keepdims=True))
    return -np.mean(np.diag(log_softmax))
```

Three things worth noticing:

1. **L2-normalise before the dot product.** Otherwise the "similarity" is just an inner product whose magnitude depends on the vector lengths, not on their direction.
2. **Subtract the row-max before the softmax.** This is a standard log-sum-exp trick — exponentials of large positive numbers overflow, but subtracting the max gives the same softmax with stable arithmetic.
3. **The target is the diagonal.** Row $i$ of the logits matrix is "anchor $i$ vs every positive in the batch"; the right answer is "positive $i$", which is column $i$. Categorical cross-entropy with that target is exactly $-\log P(i \mid i) = -\log\text{softmax}(\text{logits})_{ii}$, averaged over $i$.
</details>